# 13. 서포트 벡터 머신 타이타닉 분류 (SVM Titanic)

타이타닉 데이터셋으로 선형 커널(`linear`)과 비선형 커널(`rbf`) 기반 SVM(`SVC`) 모델을 학습하고, 데이터 누수 없는 전처리 파이프라인과 분류 성능을 비교합니다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 로컬 캐시(titanic.csv) 우선 로드, 없을 경우 seaborn 내장 데이터셋 사용
candidates = [
    Path('Study_Python/[8] Model_Evaluation/data/titanic.csv'),
    Path('../[8] Model_Evaluation/data/titanic.csv'),
    Path('data/titanic.csv'),
]
csv_path = next((p for p in candidates if p.is_file()), None)
df = pd.read_csv(csv_path) if csv_path else sns.load_dataset('titanic')
df.head()

In [ ]:
# 1. 필요한 특성과 타겟 선택 후 먼저 훈련/테스트 분할 (데이터 누수 방지)
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target = 'survived'
X = df[features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. 훈련 데이터 기준 결측치 대체 및 범주형 인코딩
age_median = X_train['age'].median()
embarked_mode = X_train['embarked'].mode()[0]
for part in (X_train, X_test):
    part['age'] = part['age'].fillna(age_median)
    part['embarked'] = part['embarked'].fillna(embarked_mode)
    part['sex'] = part['sex'].map({'female': 0, 'male': 1})
    part['embarked'] = part['embarked'].map({'C': 0, 'Q': 1, 'S': 2})

# 3. 거리 기반 모델(SVM)을 위한 표준화 (훈련 데이터에만 fit)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 선형 커널 SVM 모델 학습 및 평가
svm_linear = SVC(kernel='linear', C=10, random_state=42)
svm_linear.fit(X_train_scaled, y_train)
y_pred_linear = svm_linear.predict(X_test_scaled)
print(f'Linear SVM Test Accuracy: {accuracy_score(y_test, y_pred_linear):.4f}')
print(classification_report(y_test, y_pred_linear))

In [ ]:
# 비선형 RBF 커널 SVM 모델 학습 및 혼동 행렬 시각화
svm_rbf = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm_rbf.fit(X_train_scaled, y_train)
y_pred_rbf = svm_rbf.predict(X_test_scaled)
conf_matrix = confusion_matrix(y_test, y_pred_rbf)
print(f'RBF SVM Test Accuracy: {accuracy_score(y_test, y_pred_rbf):.4f}')
print(classification_report(y_test, y_pred_rbf))

plt.figure(figsize=(5, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Survived', 'Survived'],
            yticklabels=['Not Survived', 'Survived'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - SVM (RBF)')
plt.show()